# Notebook 2 — Cohort Retention Analysis
**What is a cohort?**  
Everyone who signed up in the same month = one cohort.  
We track what % of each cohort is still active at Month 1, 2, 3…  

**Real numbers from your data:**  
- Month-1 average retention: 85.8%  
- Month-3 average retention: 47.5%  
- Month-6 average retention: 25.1%

In [ ]:
# Imports & load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline

sns.set_style('white')

FILE_PATH  = r'E:\Mavenflix project\data\Subscription Cohort Analysis Data.csv'
OUTPUT_DIR = r'E:\Mavenflix project\outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(FILE_PATH)
df['created_date']  = pd.to_datetime(df['created_date'])
df['canceled_date'] = pd.to_datetime(df['canceled_date'])
SNAPSHOT = pd.Timestamp('2023-09-08')

df['is_churned']  = df['canceled_date'].notna().astype(int)
df['tenure_days'] = (df['canceled_date'].fillna(SNAPSHOT) - df['created_date']).dt.days
df['cohort_month']= df['created_date'].dt.to_period('M')
df['is_paid']     = (df['was_subscription_paid'] == 'Yes').astype(int)
df['end_date']    = df['canceled_date'].fillna(SNAPSHOT)

print('Data loaded successfully!')
print('Shape:', df.shape)
print('Cohorts found:', df['cohort_month'].nunique())

In [ ]:
# Build the cohort retention matrix 

# Step 1: How many months did each subscriber stay?
df['months_active'] = (
    (df['end_date'].dt.to_period('M') - df['cohort_month'])
    .apply(lambda x: x.n)
)

# Step 2: Original cohort sizes (how many signed up each month)
cohort_sizes = df.groupby('cohort_month')['customer_id'].count()
print('Cohort sizes:')
print(cohort_sizes.to_string())

# Step 3: Build retention for each month offset
retention_data = []
for offset in range(0, 13):
    still_here = (
        df[df['months_active'] >= offset]
        .groupby('cohort_month')['customer_id']
        .count()
    )
    pct = (still_here / cohort_sizes * 100).round(1)
    pct.name = f'Month {offset}'
    retention_data.append(pct)

cohort_matrix = pd.concat(retention_data, axis=1)

print('\nMatrix shape:', cohort_matrix.shape)
print('\nFirst 3 cohorts, first 5 months:')
print(cohort_matrix.iloc[:3, :5])

In [ ]:
# ── CELL 3 ── Draw the cohort HEATMAP ────────────────────────────────────────
# This is the most impressive chart in the entire project.
# Green = high retention, Red = low retention
# Month 0 is always 100% (everyone is active when they sign up)
# Watch how fast it turns red — that is the churn story.

fig, ax = plt.subplots(figsize=(18, 8))

sns.heatmap(
    cohort_matrix,
    annot=True,           # show the number inside each cell
    fmt='.1f',            # 1 decimal place e.g. 85.8
    cmap='RdYlGn',        # Red → Yellow → Green colour scale
    vmin=0, vmax=100,     # scale from 0% to 100%
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Retention Rate (%)', 'shrink': 0.8},
    annot_kws={'size': 9}
)

ax.set_title(
    'MavenFlix — Cohort Retention Heatmap\n'
    '% of each month\'s subscribers still active after N months',
    fontsize=14, fontweight='bold', pad=20
)
ax.set_xlabel('Months Since Signup', fontsize=12, labelpad=10)
ax.set_ylabel('Subscriber Signup Cohort', fontsize=12, labelpad=10)
ax.tick_params(axis='y', rotation=0)
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'chart5_cohort_heatmap.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Cohort heatmap saved to outputs folder!')

In [ ]:
# ── CELL 4 ── Retention summary numbers ─────────────────────────────────────
# These are the numbers you put in your README and say in interviews.

avg_retention = cohort_matrix.mean().round(1)

print('=== AVERAGE RETENTION BY MONTH (all cohorts averaged) ===')
for month, pct in avg_retention.items():
    bar = '█' * int(pct / 5)
    print(f'  {month:<10}: {pct:>5}%  {bar}')

m1 = avg_retention['Month 1']
m3 = avg_retention['Month 3']
m6 = avg_retention['Month 6']

print()
print('=== KEY INTERVIEW NUMBERS ===')
print(f'  Month 1 retention: {m1}%  →  {100-m1:.1f}% leave within month 1')
print(f'  Month 3 retention: {m3}%  →  over half gone by month 3')
print(f'  Month 6 retention: {m6}%  →  only 1 in 4 still here at 6 months')

In [ ]:
# ── CELL 5 ── Average retention CURVE (line chart) ───────────────────────────
# The heatmap shows each cohort individually.
# This chart shows the AVERAGE decay across all cohorts — the big picture.

avg_retention = cohort_matrix.mean()

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    range(len(avg_retention)),
    avg_retention.values,
    color='#7F77DD', marker='o', linewidth=2.5, markersize=8
)
ax.fill_between(range(len(avg_retention)), avg_retention.values,
                alpha=0.15, color='#7F77DD')

for i, val in enumerate(avg_retention.values):
    ax.text(i, val + 1.5, f'{val:.1f}%', ha='center', fontsize=8.5)

for y, label in [(50, '50% mark'), (25, '25% mark')]:
    ax.axhline(y=y, color='red', linestyle='--', alpha=0.3, linewidth=1)
    ax.text(12.1, y, label, fontsize=8, color='red', va='center')

ax.set_xticks(range(len(avg_retention)))
ax.set_xticklabels([f'Month {i}' for i in range(len(avg_retention))],
                   rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 110)
ax.set_title(
    'Average Retention Curve — MavenFlix\n'
    'By Month 6, only 25% of subscribers remain on average',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_ylabel('Average Retention (%)', fontsize=11)
ax.set_xlabel('Months Since Signup', fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'chart6_avg_retention_curve.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Retention curve saved!')
print()
print('=== NOTEBOOK 2 COMPLETE ===')
print('Next: Open 03_churn_model.ipynb')